# 04 PCA against truncated SVD (ML5)

Owner: Jeevan Mandala (ML Engineer)

Same matrix, same k, same held-out customers. SVD works on the matrix as it is. PCA first subtracts each product's average purchase value, so it describes how a customer differs from the average customer.

SVD's relative error is measured on the original matrix and PCA's explained variance on the centred one. They're different measures, so we can't compare them directly.

In [7]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path

import numpy as np
import pandas as pd

from recommend import (
    load_processed, fit_svd, fit_pca, relative_error, pca_relative_error, factor_size_mb,
    product_vectors, recommend_for_customers, recommend_blend, recommend_pca,
    best_seller_baseline, hits_at_n, bootstrap_ci, timed_with_peak_memory, pca_scores_of,
)

A, customer_index, product_index, eval_pairs = load_processed()
K = int(Path("../outputs/results/chosen_k.txt").read_text())
description = product_index["Description"].to_numpy()
print("Matrix:", A.shape, "| k =", K)

Matrix: (5471, 3686) | k = 100


## Fit both models

A throw-away fit goes first so start-up costs don't land on the first model. Memory is measured with tracemalloc, which sees Python and NumPy allocations but not memory inside compiled code, so it's only a rough comparison.

In [8]:
fit_svd(A, 5); fit_pca(A, 5)  # warm-up, not timed

svd, svd_seconds, svd_peak = timed_with_peak_memory(fit_svd, A, K)
pca, pca_seconds, pca_peak = timed_with_peak_memory(fit_pca, A, K)
print(f"SVD: {svd_seconds:.2f} s, peak {svd_peak:.1f} MB")
print(f"PCA: {pca_seconds:.2f} s, peak {pca_peak:.1f} MB")

SVD: 0.28 s, peak 17.0 MB
PCA: 0.44 s, peak 14.6 MB


## What is lost

In [9]:
svd_error = relative_error(A, svd)
pca_error = pca_relative_error(A, pca)
pca_variance = float(pca.explained_variance_ratio_.sum())

svd_mb = factor_size_mb(A, K)
pca_mb = svd_mb + A.shape[1] * 8 / 1024**2  # PCA also stores the column means

print(f"SVD relative error {svd_error:.4f}  (energy kept {(1 - svd_error**2) * 100:.1f}%)")
print(f"PCA variance explained {pca_variance * 100:.1f}%  | relative error on centred data {pca_error:.4f}")
print(f"Factors: SVD {svd_mb:.2f} MB, PCA {pca_mb:.2f} MB")

SVD relative error 0.7275  (energy kept 47.1%)
PCA variance explained 44.9%  | relative error on centred data 0.7424
Factors: SVD 6.99 MB, PCA 7.01 MB


## Hit rate on held-out purchases

Each model is scored the two ways from notebook 03. For PCA, reconstruct adds the column mean back.

In [10]:
eval_rows = np.sort(eval_pairs["customer_idx"].unique())

def hit(recs):
    return hits_at_n(recs, eval_rows, eval_pairs, A.shape)

scored = {
    ("SVD", "reconstruct"): hit(recommend_blend(A, svd, eval_rows, mode="reconstruct", alpha=1.0)),
    ("PCA", "reconstruct"): hit(recommend_pca(A, pca, eval_rows, mode="reconstruct")),
    ("SVD", "cosine"): hit(recommend_for_customers(A, product_vectors(svd), eval_rows)),
    ("PCA", "cosine"): hit(recommend_pca(A, pca, eval_rows, mode="cosine")),
    ("Best sellers", "baseline"): hit(best_seller_baseline(A, eval_rows)),
}
rows = []
for (method, scoring), h in scored.items():
    low, high = bootstrap_ci(h)
    rows.append({"method": method, "scoring": scoring, "hit_rate_at_10": h.mean(), "ci_low": low, "ci_high": high})
hit_table = pd.DataFrame(rows).round(4)
hit_table

,method,scoring,hit_rate_at_10,ci_low,ci_high
0,SVD,reconstruct,0.3832,0.3617,0.4072
1,PCA,reconstruct,0.3853,0.3638,0.4083
2,SVD,cosine,0.2253,0.2059,0.2426
3,PCA,cosine,0.2337,0.2143,0.2520
4,Best sellers,baseline,0.2703,0.2504,0.2896


## Component 1

The products with the largest loadings (both ends, since the sign is arbitrary), and how each model's component 1 score lines up with simple facts about the customer.

In [11]:
def show_ends(component, n=8):
    order = np.argsort(component)
    print("  largest positive loadings:")
    for j in order[::-1][:n]:
        print(f"    {component[j]:+.3f}  {description[j]}")
    print("  largest negative loadings:")
    for j in order[:n]:
        print(f"    {component[j]:+.3f}  {description[j]}")

print("SVD component 1"); show_ends(svd.components_[0])
print("\nPCA component 1"); show_ends(pca.components_[0])

volume = np.asarray(A.sum(axis=1)).ravel()   # total purchase value per customer
breadth = A.getnnz(axis=1)                    # number of different products bought
svd_c1 = svd.transform(A)[:, 0]
pca_c1 = pca_scores_of(pca, A)[:, 0]
corr = pd.DataFrame({
    "corr with total purchase value": [abs(np.corrcoef(c, volume)[0, 1]) for c in (svd_c1, pca_c1)],
    "corr with number of products":   [abs(np.corrcoef(c, breadth)[0, 1]) for c in (svd_c1, pca_c1)],
}, index=["SVD component 1", "PCA component 1"]).round(3)
corr

SVD component 1
  largest positive loadings:
    +0.130  PACK OF 72 RETRO SPOT CAKE CASES
    +0.122  WHITE HANGING HEART T-LIGHT HOLDER
    +0.108  JUMBO BAG RED RETROSPOT
    +0.101  LUNCH BAG RED RETROSPOT
    +0.092  60 TEATIME FAIRY CAKE CASES
    +0.088  ASSORTED COLOUR BIRD ORNAMENT
    +0.085  PACK OF 60 PINK PAISLEY CAKE CASES
    +0.081  LUNCH BAG SPACEBOY DESIGN
  largest negative loadings:
    +0.000  AMBER GLASS TASSLE BAG CHARM
    +0.000  SILVER LARIAT 40CM
    +0.000  PINK SQUARE COMPACT MIRROR
    +0.000  PURPLE GLASS TASSLE BAG CHARM
    +0.000  S/6 PAPER BOXES BLOSSOM IMAGES
    +0.000  FOUR PICTURE WHITE GLASS FRAME
    +0.000  MARAKESH LANTERN LARGE
    +0.000  S/6 PAPER BOXES ROMANTIC IMAGES

PCA component 1
  largest positive loadings:
    +0.123  PACK OF 72 RETRO SPOT CAKE CASES
    +0.106  JUMBO BAG RED RETROSPOT
    +0.104  LUNCH BAG RED RETROSPOT
    +0.094  WHITE HANGING HEART T-LIGHT HOLDER
    +0.087  60 TEATIME FAIRY CAKE CASES
    +0.083  LUNCH BAG SPACE

,corr with total purchase value,corr with number of products
SVD component 1,0.968,0.890
PCA component 1,0.968,0.886


## Comparison table

In [12]:
r_svd, r_pca = corr.loc["SVD component 1"], corr.loc["PCA component 1"]
comp1_svd = (f"Overall volume: how much and how widely a customer buys. The top products are the "
             f"best-selling lines, and the customer score follows total purchase value (|r| = {r_svd.iloc[0]:.2f}) "
             f"and number of products (|r| = {r_svd.iloc[1]:.2f}).")
comp1_pca = (f"The same direction, measured around the average customer. Big buyers dominate the variance, "
             f"so it still ranks customers by how much they buy (|r| = {r_pca.iloc[0]:.2f} and {r_pca.iloc[1]:.2f}).")

comparison = pd.DataFrame({
    "measure": [
        "k used",
        "Fit time (seconds)",
        "Peak memory during fit (MB, tracemalloc)",
        "Size of stored factors (MB)",
        "Variance explained (PCA) / matrix energy kept (SVD), %",
        "Relative reconstruction error",
        "Hit rate at 10, reconstruct scoring",
        "Hit rate at 10, cosine scoring",
        "Best-seller baseline hit rate at 10",
        "What component 1 represents, in words",
    ],
    "PCA": [
        K, round(pca_seconds, 2), round(pca_peak, 1), round(pca_mb, 2),
        round(pca_variance * 100, 1), round(pca_error, 4),
        round(scored[("PCA", "reconstruct")].mean(), 4), round(scored[("PCA", "cosine")].mean(), 4),
        round(scored[("Best sellers", "baseline")].mean(), 4),
        comp1_pca,
    ],
    "Truncated SVD": [
        K, round(svd_seconds, 2), round(svd_peak, 1), round(svd_mb, 2),
        round((1 - svd_error**2) * 100, 1), round(svd_error, 4),
        round(scored[("SVD", "reconstruct")].mean(), 4), round(scored[("SVD", "cosine")].mean(), 4),
        round(scored[("Best sellers", "baseline")].mean(), 4),
        comp1_svd,
    ],
})
comparison.to_csv("../outputs/results/pca_vs_svd.csv", index=False)
hit_table.to_csv("../outputs/results/pca_vs_svd_hit_rates.csv", index=False)
comparison

,measure,PCA,Truncated SVD
0,k used,100,100
1,Fit time (seconds),0.44,0.28
2,"Peak memory during fit (MB, tracemalloc)",14.6,17.0
3,Size of stored factors (MB),7.01,6.99
4,Variance explained (PCA) / matrix energy kept ...,44.9,47.1
5,Relative reconstruction error,0.7424,0.7275
6,"Hit rate at 10, reconstruct scoring",0.3853,0.3832
7,"Hit rate at 10, cosine scoring",0.2337,0.2253
8,Best-seller baseline hit rate at 10,0.2703,0.2703
9,"What component 1 represents, in words","The same direction, measured around the averag...",Overall volume: how much and how widely a cust...


## What this shows

The two methods come out almost the same. For both, the top component 1 products are the same popular lines (cake cases, hanging heart holders, retro spot bags), and the component 1 score follows total purchase value (|r| = 0.968) and the number of products bought. So component 1 is how much and how widely a customer buys. The negative loadings print as 0.000, so there's no real opposite end.

Centring would turn every zero into a small negative number, but the arpack solver centres implicitly, so PCA is still feasible at this size.

PCA reconstruct (0.385) and SVD reconstruct (0.383) are tied, well inside the intervals, and so are the cosine scorings (0.234 and 0.225). Centring doesn't change which products get recommended in any useful way. PCA's 44.9% of variance explained and SVD's 72.8% relative error are on different matrices, so they can't be compared directly.

SVD stays the main model. The accuracy is the same, there's no centring step, and it works straight on the sparse matrix. PCA fits a little slower (0.40 s against 0.28 s).